# QuantView Interactive Widgets Demo

This notebook demonstrates the interactive widget controls for QuantView charts.

In [ ]:
import sys

sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import plotly.io as pio
from IPython.display import display

from quantview.notebook.widgets import (
    create_chart_dashboard,
    create_date_range_picker,
    create_equity_dashboard_controls,
    create_indicator_checklist,
    create_output_area,
    create_svi_jw_sliders,
    create_svi_sliders,
    create_symbol_dropdown,
    create_vol_surface_controls,
    create_yield_curve_controls,
)

pio.renderers.default = "notebook"
print("QuantView Widgets Demo")

## Basic Input Widgets

In [ ]:
# Symbol dropdown with callback
selected_symbol = {"value": "SPY"}


def on_symbol_change(change):
    selected_symbol["value"] = change["new"]
    print(f"Symbol changed to: {change['new']}")


symbol_dropdown = create_symbol_dropdown(
    ["SPY", "QQQ", "AAPL", "MSFT", "GOOGL", "TSLA", "NVDA"],
    value="SPY",
    on_change=on_symbol_change,
)
display(symbol_dropdown)

In [ ]:
# Date range picker
date_range = create_date_range_picker(
    start=pd.Timestamp("2024-01-01").date(),
    end=pd.Timestamp("2024-12-31").date(),
    on_change=lambda c: print(f"Date range: {c}"),
)
display(date_range)

In [ ]:
# Indicator checklist
indicators = create_indicator_checklist(
    value=["sma_20", "sma_50", "bbands", "rsi"],
    on_change=lambda c: print(f"Indicators: {c['new']}"),
)
display(indicators)

## SVI Parameter Sliders

In [ ]:
# Raw SVI parameters
svi_params = {"value": {}}


def on_svi_change(params):
    svi_params["value"] = params
    print(
        f"SVI params updated: a={params['a']:.4f}, b={params['b']:.4f}, ρ={params['rho']:.4f}, m={params['m']:.4f}, σ={params['sigma']:.4f}"
    )


svi_sliders = create_svi_sliders(on_change=on_svi_change)
display(svi_sliders)

In [ ]:
# SVI Jump-Wings (trader-friendly) parameters
jw_params = {"value": {}}


def on_jw_change(params):
    jw_params["value"] = params
    print(
        f"JW params: ATM Var={params['atm_variance']:.4f}, Skew={params['atm_skew']:.4f}, Put Wing={params['put_wing_slope']:.4f}, Call Wing={params['call_wing_slope']:.4f}"
    )


jw_sliders = create_svi_jw_sliders(on_change=on_jw_change)
display(jw_sliders)

## Chart Control Panels

In [ ]:
# Volatility surface controls
vol_controls = create_vol_surface_controls(on_change=lambda p: print(f"Vol controls: {p}"))
display(vol_controls)

In [ ]:
# Yield curve controls
yc_controls = create_yield_curve_controls(on_change=lambda p: print(f"YC controls: {p}"))
display(yc_controls)

In [ ]:
# Equity dashboard controls
equity_controls = create_equity_dashboard_controls(
    ["SPY", "QQQ", "AAPL", "MSFT", "GOOGL", "TSLA", "NVDA", "META"],
    on_change=lambda p: print(f"Equity controls: {p}"),
)
display(equity_controls)

## Complete Dashboard

In [ ]:
# Build full dashboard
equity_ctrl = create_equity_dashboard_controls(["SPY", "QQQ", "AAPL"])
vol_ctrl = create_vol_surface_controls()
yc_ctrl = create_yield_curve_controls()
output_area = create_output_area()

dashboard = create_chart_dashboard(equity_ctrl, vol_ctrl, yc_ctrl, output_area)
display(dashboard)

## Using Widgets with Charts

Example of linking widgets to chart updates.

In [ ]:
# Example: Interactive candlestick with widget controls
from quantview.notebook.charts import candlestick_chart

# Generate sample data
np.random.seed(42)
dates = pd.date_range("2024-01-01", periods=200, freq="B")
returns = np.random.normal(0.0005, 0.015, len(dates))
prices = 100 * np.exp(np.cumsum(returns))
noise = np.random.uniform(0.995, 1.005, (len(dates), 4))
df = pd.DataFrame(
    {
        "open": prices * noise[:, 0],
        "high": prices * noise[:, 1],
        "low": prices * noise[:, 2],
        "close": prices * noise[:, 3],
        "volume": np.random.lognormal(13, 0.5, len(dates)).astype(int),
    },
    index=dates,
)
df["high"] = df[["open", "high", "close"]].max(axis=1)
df["low"] = df[["open", "low", "close"]].min(axis=1)

# Create initial chart
initial_fig = candlestick_chart(df, "SPY", indicators=["sma_20", "sma_50", "bbands"])
initial_fig.show()

In [ ]:
# Create controls for the chart
controls = create_equity_dashboard_controls(
    ["SPY"], on_change=lambda p: print(f"Update chart with: {p}")
)
display(controls)

# In a real app, you would:
# 1. Fetch data for selected symbol/date range
# 2. Recreate chart with selected indicators
# 3. Update output_area with new figure
#
# with output_area:
#     output_area.clear_output()
#     new_fig = candlestick_chart(new_data, symbol, indicators=selected_indicators)
#     display(new_fig)

## Widget Styling Customization

In [ ]:
# Custom styling example
from quantview.notebook.widgets import CONFIG

print("Current widget config:")
print(f"  Layout width: {CONFIG.layout_width}")
print(f"  Continuous update: {CONFIG.continuous_update}")
print(f"  Style: {CONFIG.style}")

# Modify globally
CONFIG.layout_width = "400px"
CONFIG.style["description_width"] = "150px"

# New widgets will use updated config
custom_dropdown = create_symbol_dropdown(["A", "B", "C"], description="Custom:")
display(custom_dropdown)

## Export Widget State